In [1]:
import numpy as np
import pandas as pd


# =====================================================================
# 1. DATA LOADING & SPLITTING
# =====================================================================
def load_and_split_data(
    url: str, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Loads a CSV from a URL and performs a 60/20/20 train/val/test split."""
    df = pd.read_csv(url)

    # Shuffle index deterministically
    np.random.seed(seed)
    n = len(df)
    idx = np.arange(n)
    np.random.shuffle(idx)

    # Compute split boundaries
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    # Split dataframe
    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train : n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val :]].reset_index(drop=True)

    return df_train, df_val, df_test


# =====================================================================
# 2. CATEGORICAL ONE-HOT ENCODING (LEAK-FREE)
# =====================================================================
def fit_categorical_encoder(
    df_train: pd.DataFrame, cat_cols: list[str]
) -> dict[str, list]:
    """Learns unique categories for each categorical column exclusively from the training set."""
    encoding_map = {}
    for col in cat_cols:
        unique_cats = sorted(df_train[col].dropna().unique().tolist())
        encoding_map[col] = unique_cats
    return encoding_map


def transform_categorical(
    df: pd.DataFrame, encoding_map: dict[str, list]
) -> pd.DataFrame:
    """Transforms categorical columns into one-hot binary flags using pre-learned categories."""
    encoded_dfs = []

    for col, categories in encoding_map.items():
        col_series = df[col]
        for cat in categories:
            feature_name = f"{col}_{cat}"
            encoded_dfs.append((col_series == cat).astype(int).rename(feature_name))

    if encoded_dfs:
        return pd.concat(encoded_dfs, axis=1)
    return pd.DataFrame(index=df.index)


# =====================================================================
# 3. MISSING VALUE IMPUTATION (LEAK-FREE)
# =====================================================================
def fit_imputer(
    df_train: pd.DataFrame, num_cols: list[str], strategy: str = "median"
) -> dict[str, float]:
    """Computes summary statistics (mean or median) for numerical columns on the training set."""
    imputation_values = {}
    for col in num_cols:
        if strategy == "median":
            val = df_train[col].median()
        elif strategy == "mean":
            val = df_train[col].mean()
        else:
            raise ValueError(f"Unsupported strategy: {strategy}")
        imputation_values[col] = float(val)
    return imputation_values


def transform_imputer(
    df: pd.DataFrame, imputation_values: dict[str, float]
) -> pd.DataFrame:
    """Imputes missing values in numerical columns using pre-computed statistics."""
    df_num = df[list(imputation_values.keys())].copy()
    return df_num.fillna(imputation_values)


# =====================================================================
# 4. FEATURE SCALING (LEAK-FREE)
# =====================================================================
def fit_transform_scaler(
    df_train_num: pd.DataFrame,
    df_val_num: pd.DataFrame,
    df_test_num: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Standardizes numerical features (zero mean, unit variance) using training set statistics."""
    means = df_train_num.mean()
    stds = df_train_num.std().replace(0, 1.0)  # Prevent division by zero

    train_scaled = (df_train_num - means) / stds
    val_scaled = (df_val_num - means) / stds
    test_scaled = (df_test_num - means) / stds

    return train_scaled, val_scaled, test_scaled


# =====================================================================
# 5. TARGET TRANSFORMATIONS (LOG1P / EXPM1)
# =====================================================================
def transform_target_log(y: np.ndarray) -> np.ndarray:
    """Applies log(1 + y) transformation to reduce skewness in right-skewed target distributions."""
    return np.log1p(y)


def inverse_transform_target(y_log: np.ndarray) -> np.ndarray:
    """Applies exp(y) - 1 inverse transformation to restore predictions to the original scale."""
    return np.expm1(y_log)


# =====================================================================
# 6. FEATURE MATRIX ASSEMBLY
# =====================================================================
def prepare_feature_matrix(
    df_num: pd.DataFrame, df_cat_encoded: pd.DataFrame, add_bias: bool = True
) -> tuple[np.ndarray, list[str]]:
    """Combines processed numerical and categorical features into a single NumPy array with a bias column.

    Returns the array along with the ordered list of feature names.
    """
    features = [df_num.values, df_cat_encoded.values]
    feature_names = list(df_num.columns) + list(df_cat_encoded.columns)

    if add_bias:
        bias_column = np.ones((len(df_num), 1))
        features.insert(0, bias_column)
        feature_names.insert(0, "bias_intercept")

    X = np.hstack(features)
    return X, feature_names


# =====================================================================
# 7. RIDGE REGRESSION SOLVER (PURE NUMPY)
# =====================================================================
def train_ridge_regression(
    X: np.ndarray, y: np.ndarray, r: float = 0.0
) -> np.ndarray:
    """Fits Ridge Regression using Normal Equations: w = (X^T X + r*I)^(-1) X^T y.

    Excludes bias column (index 0) from regularization penalty.
    """
    n_features = X.shape[1]
    XTX = X.T @ X

    # Create Identity matrix and zero out bias penalty
    I = np.eye(n_features)
    I[0, 0] = 0.0

    XTX_reg = XTX + r * I
    weights = np.linalg.inv(XTX_reg) @ X.T @ y

    return weights


def predict(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Computes predictions given feature matrix X and weight vector."""
    return X @ weights


# =====================================================================
# 8. FEATURE IMPORTANCE & WEIGHT INSPECTION
# =====================================================================
def get_feature_importances(
    weights: np.ndarray, feature_names: list[str], top_n: int = 5
) -> pd.DataFrame:
    """Extracts, ranks, and prints top positive and negative learned weight coefficients."""
    df_weights = pd.DataFrame({
        "feature": feature_names,
        "weight": weights,
        "abs_weight": np.abs(weights),
    })

    # Separate bias from feature evaluation
    df_features = df_weights[df_weights["feature"] != "bias_intercept"].copy()

    top_positives = (
        df_features.sort_values(by="weight", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    top_negatives = (
        df_features.sort_values(by="weight", ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )

    print("=" * 60)
    print("FEATURE IMPORTANCE ANALYSIS (RIDGE WEIGHT COEFFICIENTS)")
    print("=" * 60)

    bias_val = df_weights.loc[
        df_weights["feature"] == "bias_intercept", "weight"
    ].values
    if len(bias_val) > 0:
        print(f"Bias / Intercept Weight: {bias_val[0]:+.5f}\n")

    print(f"--- Top {top_n} Positive Features (Increases Target) ---")
    for _, row in top_positives.iterrows():
        print(f"{row['feature']:<35} : {row['weight']:+.5f}")

    print(f"\n--- Top {top_n} Negative Features (Decreases Target) ---")
    for _, row in top_negatives.iterrows():
        print(f"{row['feature']:<35} : {row['weight']:+.5f}")

    print("=" * 60 + "\n")

    return df_features.sort_values(
        by="abs_weight", ascending=False
    ).reset_index(drop=True)


# =====================================================================
# 9. EVALUATION & EXPERIMENTS
# =====================================================================
def calculate_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Computes Root Mean Squared Error (RMSE) on the original scale."""
    error = y_true - y_pred
    mse = np.mean(error**2)
    return float(np.sqrt(mse))


def evaluate_regularization(
    X_train: np.ndarray,
    y_train_orig: np.ndarray,
    X_val: np.ndarray,
    y_val_orig: np.ndarray,
    r_values: list[float],
    use_log_target: bool = True,
) -> dict[float, float]:
    """Evaluates validation RMSE across multiple regularization parameters, converting back to original scale if log transform is used."""
    results = {}

    y_train_fit = (
        transform_target_log(y_train_orig) if use_log_target else y_train_orig
    )

    for r in r_values:
        w = train_ridge_regression(X_train, y_train_fit, r=r)
        y_pred_raw = predict(X_val, w)

        y_pred_final = (
            inverse_transform_target(y_pred_raw)
            if use_log_target
            else y_pred_raw
        )

        rmse = calculate_rmse(y_val_orig, y_pred_final)
        results[r] = rmse

    return results


def run_seed_experiment(
    url: str,
    target_col: str,
    cat_cols: list[str],
    num_cols: list[str],
    seeds: list[int],
    r: float = 0.0,
    use_scaler: bool = True,
    use_log_target: bool = True,
) -> tuple[float, float, list[float]]:
    """Runs the full pipeline across multiple random seeds to measure stability with all preprocessing options enabled."""
    rmse_scores = []

    for seed in seeds:
        # 1. Load & Split
        df_train, df_val, df_test = load_and_split_data(url, seed=seed)

        # Extract Raw Targets
        y_train_orig = df_train[target_col].values
        y_val_orig = df_val[target_col].values

        y_train_fit = (
            transform_target_log(y_train_orig)
            if use_log_target
            else y_train_orig
        )

        # 2. Imputation
        imp_map = fit_imputer(df_train, num_cols, strategy="median")
        X_train_num = transform_imputer(df_train, imp_map)
        X_val_num = transform_imputer(df_val, imp_map)
        X_test_num = transform_imputer(df_test, imp_map)

        # 3. Scaling
        if use_scaler:
            X_train_num, X_val_num, _ = fit_transform_scaler(
                X_train_num, X_val_num, X_test_num
            )

        # 4. Encoding
        enc_map = fit_categorical_encoder(df_train, cat_cols)
        X_train_cat = transform_categorical(df_train, enc_map)
        X_val_cat = transform_categorical(df_val, enc_map)

        # 5. Feature Assembly
        X_train, _ = prepare_feature_matrix(X_train_num, X_train_cat)
        X_val, _ = prepare_feature_matrix(X_val_num, X_val_cat)

        # 6. Fit Model & Predict
        weights = train_ridge_regression(X_train, y_train_fit, r=r)
        y_pred_raw = predict(X_val, weights)

        # 7. Invert Transformation & Evaluate
        y_pred_final = (
            inverse_transform_target(y_pred_raw)
            if use_log_target
            else y_pred_raw
        )
        rmse = calculate_rmse(y_val_orig, y_pred_final)

        rmse_scores.append(rmse)

    mean_rmse = float(np.mean(rmse_scores))
    std_rmse = float(np.std(rmse_scores))

    return mean_rmse, std_rmse, rmse_scores


# =====================================================================
# MAIN PIPELINE EXECUTION
# =====================================================================
if __name__ == "__main__":
    DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

    # 1. Dataset Configuration & Type Detection
    df_raw = pd.read_csv(DATA_URL)
    TARGET_COL = "median_house_value"

    feature_cols = [c for c in df_raw.columns if c != TARGET_COL]

    NUM_COLS = (
        df_raw[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    )
    CAT_COLS = (
        df_raw[feature_cols]
        .select_dtypes(exclude=[np.number])
        .columns.tolist()
    )

    print("--- Dataset Setup ---")
    print(f"Target: '{TARGET_COL}'")
    print(f"Numerical Features ({len(NUM_COLS)}): {NUM_COLS}")
    print(f"Categorical Features ({len(CAT_COLS)}): {CAT_COLS}\n")

    # Pipeline Settings
    USE_SCALER = True
    USE_LOG_TARGET = True

    # 2. Data Loading & Splitting (Seed 42)
    df_train, df_val, df_test = load_and_split_data(DATA_URL, seed=42)

    # Extract Target Vectors (Raw Original Scale)
    y_train_orig = df_train[TARGET_COL].values
    y_val_orig = df_val[TARGET_COL].values
    y_test_orig = df_test[TARGET_COL].values

    y_train_fit = (
        transform_target_log(y_train_orig) if USE_LOG_TARGET else y_train_orig
    )

    # 3. Missing Value Imputation
    imputation_map = fit_imputer(df_train, NUM_COLS, strategy="median")
    df_train_num = transform_imputer(df_train, imputation_map)
    df_val_num = transform_imputer(df_val, imputation_map)
    df_test_num = transform_imputer(df_test, imputation_map)

    # 4. Feature Scaling
    if USE_SCALER:
        df_train_num, df_val_num, df_test_num = fit_transform_scaler(
            df_train_num, df_val_num, df_test_num
        )

    # 5. Categorical One-Hot Encoding
    encoding_map = fit_categorical_encoder(df_train, CAT_COLS)
    df_train_cat = transform_categorical(df_train, encoding_map)
    df_val_cat = transform_categorical(df_val, encoding_map)
    df_test_cat = transform_categorical(df_test, encoding_map)

    # 6. Feature Matrix Assembly (Returning matrix & aligned feature names)
    X_train, feature_names = prepare_feature_matrix(
        df_train_num, df_train_cat
    )
    X_val, _ = prepare_feature_matrix(df_val_num, df_val_cat)
    X_test, _ = prepare_feature_matrix(df_test_num, df_test_cat)

    # 7. Regularization Search
    r_candidates = [0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    val_results = evaluate_regularization(
        X_train,
        y_train_orig,
        X_val,
        y_val_orig,
        r_candidates,
        use_log_target=USE_LOG_TARGET,
    )

    print("--- Regularization Search (Validation RMSE) ---")
    best_r = 0.0
    best_val_rmse = float("inf")
    for r_val, rmse in val_results.items():
        print(f"r = {r_val:<7} | Val RMSE = ${rmse:,.2f}")
        if rmse < best_val_rmse:
            best_val_rmse = rmse
            best_r = r_val

    # 8. Final Model Fitting on Train + Validation
    X_full_train = np.vstack([X_train, X_val])
    y_full_train_orig = np.concatenate([y_train_orig, y_val_orig])
    y_full_train_fit = (
        transform_target_log(y_full_train_orig)
        if USE_LOG_TARGET
        else y_full_train_orig
    )

    final_weights = train_ridge_regression(
        X_full_train, y_full_train_fit, r=best_r
    )

    # --- FEATURE IMPORTANCE INSPECTION ---
    df_importance = get_feature_importances(
        final_weights, feature_names, top_n=5
    )

    # 9. Final Evaluation on Holdout Test Set
    y_test_pred_raw = predict(X_test, final_weights)
    y_test_pred_final = (
        inverse_transform_target(y_test_pred_raw)
        if USE_LOG_TARGET
        else y_test_pred_raw
    )
    test_rmse = calculate_rmse(y_test_orig, y_test_pred_final)

    print(f"Optimal r: {best_r} | Holdout Test Set RMSE: ${test_rmse:,.2f}\n")

    # 10. Multi-Seed Stability Experiment
    seeds_to_test = [10, 20, 30, 42, 100, 2026]

    mean_rmse, std_rmse, per_seed_scores = run_seed_experiment(
        url=DATA_URL,
        target_col=TARGET_COL,
        cat_cols=CAT_COLS,
        num_cols=NUM_COLS,
        seeds=seeds_to_test,
        r=best_r,
        use_scaler=USE_SCALER,
        use_log_target=USE_LOG_TARGET,
    )

    print("--- Multi-Seed Stability Experiment ---")
    for seed, score in zip(seeds_to_test, per_seed_scores):
        print(f"Seed {seed:<5} -> Val RMSE: ${score:,.2f}")

    print(f"\nFinal Stability Metrics Across {len(seeds_to_test)} Seeds:")
    print(f"Mean RMSE: ${mean_rmse:,.2f}")
    print(f"Std Dev:   ${std_rmse:,.2f}")

--- Dataset Setup ---
Target: 'median_house_value'
Numerical Features (8): ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical Features (1): ['ocean_proximity']

--- Regularization Search (Validation RMSE) ---
r = 0.0     | Val RMSE = $235,970.41
r = 0.0001  | Val RMSE = $82,790.17
r = 0.001   | Val RMSE = $82,790.12
r = 0.01    | Val RMSE = $82,789.55
r = 0.1     | Val RMSE = $82,784.22
r = 1.0     | Val RMSE = $82,751.93
r = 10.0    | Val RMSE = $82,737.45
r = 100.0   | Val RMSE = $82,870.07
FEATURE IMPORTANCE ANALYSIS (RIDGE WEIGHT COEFFICIENTS)
Bias / Intercept Weight: +12.14226

--- Top 5 Positive Features (Increases Target) ---
median_income                       : +0.31374
ocean_proximity_ISLAND              : +0.18138
households                          : +0.16338
total_bedrooms                      : +0.09882
ocean_proximity_<1H OCEAN           : +0.05110

--- Top 5 Negative Features (Decreas